# 04 — Logistic Regression benchmark for machine-failure classification

**Objective.** Establish a classification-specific benchmark using the same six predictors and 8,000 development observations, while measuring both alarm performance at a stated threshold and threshold-independent ranking quality.

**Workflow.** Recreate the approved split → fit scaling/encoding inside a pipeline → obtain five-fold OOF class-1 scores → report classification and ranking metrics → interpret minority-class detection.

`predict_proba(... )[:, 1]` provides the model's estimated class-1 probability. It is not a field-validated guarantee of actual future failure risk.

## 1. Load and recreate the development split

Feature choices, train/test splitting and model logic are all written in this notebook, so you can understand every step without another file.

In [13]:
from IPython.display import display
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score,
    recall_score, f1_score, confusion_matrix,
)
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from pathlib import Path
from sklearn.model_selection import train_test_split

# Exact data schema used throughout the original experiments.
RANDOM_STATE = 42
TARGET = "Machine failure"
CATEGORICAL_FEATURES = ["Type"]
NUMERIC_FEATURES = [
    "Air temperature [K]", "Process temperature [K]",
    "Rotational speed [rpm]", "Torque [Nm]", "Tool wear [min]",
]
FEATURES = CATEGORICAL_FEATURES + NUMERIC_FEATURES

# Accept execution from the repository root or its notebooks/ directory.
# No extra .py helper file is needed.
DATA_PATH = Path("data/raw/ai4i2020.csv")
if not DATA_PATH.is_file():
    DATA_PATH = Path("../data/raw/ai4i2020.csv")
if not DATA_PATH.is_file():
    raise FileNotFoundError(
        "Cannot locate data/raw/ai4i2020.csv. Open the project root "
        "or its notebooks/ folder in VS Code and keep the original CSV in data/raw/. "
        f"Current working directory: {Path.cwd()}"
    )

df = pd.read_csv(DATA_PATH)
print(f"Dataset loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")


Dataset loaded: 10,000 rows × 14 columns


In [14]:
# These six inputs are available at prediction time; failure-mode flags and IDs
# are deliberately excluded because they could leak the target or identify rows.
required_columns = FEATURES + [TARGET]
missing_columns = [name for name in required_columns if name not in df.columns]
if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")
if df[required_columns].isna().any().any():
    raise ValueError("Missing values found in the selected modelling columns.")

X = df[FEATURES].copy()
y = df[TARGET].copy()

# Use precisely the same stratified, seeded 80/20 row split in every notebook.
# Stratification maintains approximately 3.39% failures in each subset.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y,
)

print(f"Development set: {len(X_train):,}")
print(f"Previously inspected held-out test set: {len(X_test):,}")
print(f"Development failure rate: {100 * y_train.mean():.2f}%")
print(f"Test failure rate: {100 * y_test.mean():.2f}%")

Development set: 8,000
Previously inspected held-out test set: 2,000
Development failure rate: 3.39%
Test failure rate: 3.40%


## 2. Construct the logistic pipeline

`StandardScaler` is especially relevant because logistic regression is regularised and feature magnitudes differ. Categorical `Type` is one-hot encoded. During cross-validation, preprocessing is fitted *only* on the training fold.

This is the initial, **unweighted** model (`C=1` by default, maximum 1,000 optimisation iterations).

In [15]:
# ---------------------------------------------------------
# Preprocessing
# ---------------------------------------------------------

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            StandardScaler(),
            NUMERIC_FEATURES,
        ),
        (
            "categorical",
            OneHotEncoder(
                drop="first",
                handle_unknown="ignore",
            ),
            CATEGORICAL_FEATURES,
        ),
    ]
)


# ---------------------------------------------------------
# Logistic Regression model
# ---------------------------------------------------------
# class_weight is intentionally left unchanged during
# this initial experiment.

logistic_model = LogisticRegression(
    max_iter=1000,
    random_state=RANDOM_STATE,
)


# ---------------------------------------------------------
# Complete pipeline
# ---------------------------------------------------------

logistic_pipeline = Pipeline(
    steps=[
        ("preprocessing", preprocessor),
        ("model", logistic_model),
    ]
)


# ---------------------------------------------------------
# Cross-validation
# ---------------------------------------------------------

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE,
)

print("Logistic Regression pipeline created.")
print("Cross-validation: 5-fold stratified")
print("Initial classification threshold: 0.50")

Logistic Regression pipeline created.
Cross-validation: 5-fold stratified
Initial classification threshold: 0.50


## 3. Evaluate out-of-fold failure scores

Each development record receives an OOF prediction from a fold that did not train on that record. The initial **0.50** threshold determines the binary alarm decision: changing the threshold changes the confusion matrix, precision, recall and F1, but **does not** change ROC-AUC or average precision for the same scores.

Here, “PR-AUC” refers to scikit-learn's `average_precision_score` rather than a trapezoidally integrated PR curve. Confusion-matrix rows denote actual labels; columns denote predicted labels.

In [16]:
# Every development observation receives an out-of-fold failure probability.
# Preprocessing is fitted within the training fold by the pipeline.
failure_probability_oof = cross_val_predict(
    logistic_pipeline, X_train, y_train, cv=cv, method="predict_proba",
)[:, 1]
print("Predicted failure-probability range: "
      f"{failure_probability_oof.min():.4f} to {failure_probability_oof.max():.4f}")

CLASSIFICATION_THRESHOLD = 0.50
y_pred_oof = (failure_probability_oof >= CLASSIFICATION_THRESHOLD).astype(int)

# Failure-class metrics use 1 as the positive class.
classification_metrics = pd.DataFrame({"Score": {
    "Accuracy": accuracy_score(y_train, y_pred_oof),
    "Balanced Accuracy": balanced_accuracy_score(y_train, y_pred_oof),
    "Precision": precision_score(y_train, y_pred_oof, zero_division=0),
    "Recall": recall_score(y_train, y_pred_oof, zero_division=0),
    "F1-score": f1_score(y_train, y_pred_oof, zero_division=0),
}})
# ROC-AUC and average precision use all ranking scores, not a fixed threshold.
classification_metrics.loc["ROC-AUC", "Score"] = roc_auc_score(
    y_train, failure_probability_oof,
)
classification_metrics.loc["PR-AUC", "Score"] = average_precision_score(
    y_train, failure_probability_oof,
)
print("\nCross-validated performance:")
display(classification_metrics.round(4))
print("\nCross-validated confusion matrix:")
cm = confusion_matrix(y_train, y_pred_oof, labels=[0, 1])
display(pd.DataFrame(cm,
    index=["Actual normal", "Actual failure"],
    columns=["Predicted normal", "Predicted failure"],
))
print(f"\nActual failures: {int(y_train.sum()):,}")
print(f"Predicted failures: {int(y_pred_oof.sum()):,}")

Predicted failure-probability range: 0.0001 to 0.9458

Cross-validated performance:


,Score
Accuracy,0.9704
Balanced Accuracy,0.5983
Precision,0.7297
Recall,0.1993
F1-score,0.3130
ROC-AUC,0.8932
PR-AUC,0.4589



Cross-validated confusion matrix:


,Predicted normal,Predicted failure
Actual normal,7709,20
Actual failure,217,54



Actual failures: 271
Predicted failures: 74


## Logistic Regression Findings and Engineering Implications

Logistic Regression was evaluated using five-fold cross-validation to classify normal and failure observations.

At threshold 0.50, it achieved precision = 0.7297, recall = 0.1993 and F1-score = 0.3130. It correctly identified 54 of 271 failures but missed 217, while producing 20 false alarms.

Although its ROC-AUC of 0.8932 showed useful ability to distinguish between classes, the default threshold detected only about 20% of failures.

Lowering the threshold may improve recall, but can also increase false alarms.

**Conclusion:** Logistic Regression performed better than Linear Regression for this classification task, but its failure detection remained limited. Notebook 05 investigates threshold adjustments and a nonlinear Decision Tree.